# 16S Analyses of the Longitudinal Acne Study
## Relative Abundance Plots

Date last updated: 1/1/25

Notebook authors: Yang Chen and Britta De Pessemier

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:

- 16S V1-V3 and V4 relative abundance plots at Genus taxon level aggregated by skin group
- 16S V1-V3 and V4 relative abundance plots at Genus taxon level by each sample

In [320]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
import os
from matplotlib.colors import ListedColormap
from matplotlib.colors import to_rgba


In [321]:
# Load the metadata
metadata_path = '../Metadata/metadata_final_22102024_with_age.tsv'
metadata = pd.read_csv(metadata_path, sep='\t')
metadata['severity_group'].value_counts()

severity_group
low Acne_L         70
moderate Acne_L    64
absent Healthy     57
absent Acne_NL     27
high Acne_L        25
low Acne_NL        23
Name: count, dtype: int64

In [322]:
metadata = pd.read_csv(metadata_path, sep="\t")

metadata['SampleID'] = metadata['SampleID'].astype(str)
metadata = metadata.set_index('SampleID')
metadata


,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,taxon_id,...,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group,subject_ID_x_group,age
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D14.C1,C1,not applicable,Non-lesional,skin,Day 14,not applicable,14,1,not applicable,539655,...,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy,18
LAMI.RD001.D28.C1,C1,not applicable,Non-lesional,skin,Day 28,not applicable,28,1,not applicable,539655,...,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy,18
LAMI.RD001.D0.C2,C2,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,539655,...,RD001,healthy,PP_1,PP_1C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_1_Healthy,18
LAMI.RD001.D28.C2,C2,not applicable,Non-lesional,skin,Day 28,not applicable,28,1,not applicable,539655,...,RD001,healthy,PP_1,PP_1C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_1_Healthy,18
LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,539655,...,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D16.C2,C2,not applicable,Lesional,skin,Day 16,not applicable,16,319,not applicable,539655,...,RD319,acne,PP_319,PP_319C2,Lesional_C2,Acne_L,high,high Acne_L,PP_319_Acne_L,20
LAMI.RD319.D9.C1,C1,not applicable,Lesional,skin,Day 9,not applicable,9,319,not applicable,539655,...,RD319,acne,PP_319,PP_319C1,Lesional_C1,Acne_L,moderate,moderate Acne_L,PP_319_Acne_L,20
LAMI.RD319.D2.C2,C2,not applicable,Lesional,skin,Day 2,not applicable,2,319,not applicable,539655,...,RD319,acne,PP_319,PP_319C2,Lesional_C2,Acne_L,high,high Acne_L,PP_319_Acne_L,20


In [323]:
# Define paths to the collapsed taxa tables
biom_paths = {
    'V1-V3': '../Data/16S/Tables/with_taxonomy/179426_feature-table_16S_V1V3_rare-11054_Gg2_genus-ASV.biom',
    'V4': '../Data/16S/Tables/with_taxonomy/174951_feature-table_16S_V4_rare-3769_Gg2_genus-ASV.biom'
}


In [324]:
# Predefined color palette for specific taxon
taxa_colors = {
    'g__Cutibacterium ASV-1': '#ffa505',  # Bright orange
    'g__uncultured ASV-1': '#585858',         # Dark grey
    'g__Staphylococcus ASV-1': '#92f0f0',      # Fluorescent light blue
    'g__Streptococcus ASV-1': '#e2b46c',    # Beige
    'g__Corynebacterium ASV-1': '#ffe59a',        # Pastel yellow
    'g__Lawsonella ASV-1': '#70a8dc',         # Light blue
    'g__Veillonella ASV-1': '#c5bce0',         # Pastel purplish
    'g__Micrococcus ASV-1':'#f4cccd',           # Pastel yellow
    'g__Alloprevotella ASV-1': '#8c5079',        # Plum
    'g__Lactobacillus ASV-1': '#daead3',     # Pale mint green
    'g__Neisseria ASV-1': '#f6475f',         # Redish pink
    'g__Anaerococcus ASV-1': '#008000',         # Green
    'Others': '#ededed'                 # White
}

In [325]:
# A list of unique colors to use for taxa not predefined
unique_colors = sns.color_palette("deep", n_colors=20).as_hex()
unique_color_iter = cycle(unique_colors)  # Iterator to cycle through unique colors

In [326]:
# Function to load BIOM table, collapse by taxa, sort rows by row sum, remove specified samples, and convert to relative abundance
def load_biom_table(biom_path, metadata_path):
    # Load metadata as a DataFrame from the file path
    metadata = pd.read_csv(metadata_path, sep='\t')

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))

    # Fix column sample names to match metadata
    df.columns = df.columns.str.replace(r'^.*?(LAMI\..*)$', r'\1', regex=True)

    # Remove _ before ASV to match color names
    df.index = df.index.str.replace(r'_ASV', ' ASV', regex=True)

    # Replace 'uncultured' row with 'uncultured Neisseriaceae'
    # df = df.rename(index={' g__uncultured': ' g__uncultured Neisseriaceae'})
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])
    
    # Convert the table to relative abundances
    df = df.div(df.sum(axis=0), axis=1)
    
    return df


In [327]:
# Function to determine the top n taxon and collapse the rest as "Others"
def collapse_top_n(df):
    top_taxa = df.sum(axis=1).nlargest(20).index  # Select top n taxon
    df_top = df.loc[top_taxa]
    df_top.loc['Others'] = df.loc[~df.index.isin(top_taxa)].sum()
    return df_top

In [328]:
collapse_top_n(df)

,LAMI.RD001.D0.C1,LAMI.RD001.D0.C2,LAMI.RD001.D14.C1,LAMI.RD001.D28.C1,LAMI.RD001.D28.C2,LAMI.RD004.D0.C1,LAMI.RD004.D0.C2,LAMI.RD006.D14.C2,LAMI.RD007.D14.C1,LAMI.RD007.D14.C2,...,LAMI.RD319.D21.C1,LAMI.RD319.D21.C2,LAMI.RD319.D23.C1,LAMI.RD319.D23.C2,LAMI.RD319.D25.C2,LAMI.RD319.D28.C1,LAMI.RD319.D28.C3,LAMI.RD319.D7.C3,LAMI.RD319.D9.C1,LAMI.RD319.D9.C2
g__Staphylococcus ASV-1,0.090057,0.099063,0.128612,0.165148,0.234920,0.175215,0.456985,0.216498,0.547858,0.431105,...,0.42,0.57,0.310345,0.378277,0.400000,0.179487,0.287356,0.681818,0.214953,0.200000
g__Lawsonella ASV-1,0.037216,0.061595,0.129178,0.130125,0.136414,0.014231,0.026279,0.108556,0.097106,0.129942,...,0.20,0.23,0.137931,0.329588,0.317647,0.410256,0.264368,0.090909,0.467290,0.413793
g__Streptococcus ASV-1,0.114489,0.098212,0.060623,0.081435,0.062258,0.062556,0.039004,0.017479,0.036171,0.038663,...,0.01,0.00,0.020690,0.014981,0.005882,0.008547,0.000000,0.000000,0.018692,0.013793
g__Corynebacterium ASV-1,0.007102,0.005109,0.004249,0.014522,0.022136,0.020160,0.015491,0.054278,0.064830,0.081395,...,0.06,0.05,0.020690,0.029963,0.047059,0.047009,0.149425,0.045455,0.084112,0.062069
g__Corynebacterium ASV-2,0.000000,0.012773,0.001983,0.006264,0.017432,0.084495,0.019364,0.000000,0.000000,0.000000,...,0.06,0.02,0.020690,0.029963,0.035294,0.076923,0.011494,0.045455,0.000000,0.041379
g__Acinetobacter ASV-1,0.020170,0.013057,0.007082,0.005125,0.010515,0.019567,0.012448,0.000000,0.001948,0.000000,...,0.00,0.00,0.000000,0.007491,0.005882,0.000000,0.000000,0.000000,0.000000,0.000000
g__Micrococcus ASV-1,0.028409,0.021289,0.026346,0.020501,0.021306,0.024607,0.022407,0.045998,0.010017,0.017733,...,0.03,0.01,0.048276,0.003745,0.023529,0.000000,0.011494,0.000000,0.000000,0.006897
g__Haemophilus_D_735815 ASV-1,0.113636,0.114675,0.037677,0.025057,0.022413,0.000889,0.006639,0.001533,0.018086,0.038372,...,0.00,0.00,0.000000,0.000000,0.000000,0.038462,0.011494,0.000000,0.000000,0.000000
g__Cutibacterium ASV-1,0.000284,0.000852,0.000000,0.000285,0.001107,0.001779,0.000277,0.003680,0.006121,0.011337,...,0.01,0.01,0.034483,0.011236,0.005882,0.000000,0.022989,0.022727,0.000000,0.034483
g__Gemella ASV-1,0.035795,0.033210,0.015014,0.037585,0.019369,0.009487,0.000000,0.005213,0.003617,0.006686,...,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [329]:
# Function to get or assign colors to taxon
def get_taxa_colors(taxon, global_taxa_color_map):
    for taxa in taxon:
        if taxa not in global_taxa_color_map:
            if taxa in taxa_colors:
                global_taxa_color_map[taxa] = taxa_colors[taxa]
            else:
                global_taxa_color_map[taxa] = next(unique_color_iter)  # Assign a new unique color
    return global_taxa_color_map

## Relative abundance plots

In [330]:
def plot_relative_abundance(df, metadata, group_column, output_dir, key, taxa_color_map):

    # Align samples between table and metadata
    shared_samples = df.columns.intersection(metadata.index)
    if len(shared_samples) == 0:
        raise ValueError("No overlapping samples between table and metadata.")

    df = df.loc[:, shared_samples]
    metadata = metadata.loc[shared_samples]

    # Average by group
    df_grouped = df.groupby(metadata[group_column], axis=1).mean()

    desired_order = ['Healthy', 'Acne_NL', 'Acne_L']
    present_groups = [g for g in desired_order if g in df_grouped.columns]
    df_grouped = df_grouped[present_groups]

    # Compute group sizes dynamically
    group_sizes = (
        metadata[group_column]
        .value_counts()
        .reindex(present_groups)
    )

    # Output file path
    os.makedirs(output_dir, exist_ok=True)
    output_png_file = os.path.join(
        output_dir, f'Relative_abundance_{key}_Genus-ASV.png'
    )

    # Plot title
    if key == 'V1-V3':
        plot_title = '16S V1-V3 rRNA Gg2'
    elif key == 'V4':
        plot_title = '16S V4 rRNA Gg2'
    else:
        plot_title = 'Relative Abundance'

    # Plot stacked bars
    ax = df_grouped.T.plot(
        kind='bar',
        stacked=True,
        figsize=(8.5, 9.5),
        width=0.8,
        color=[taxa_color_map.get(t, '#ADD8E6') for t in df_grouped.index]
    )

    plt.ylabel('Relative Abundance', fontsize=18)
    plt.xlabel(' ')
    plt.title(plot_title, fontsize=18)

    # Dynamic x-axis labels
    label_map = {
        'Healthy': 'H',
        'Acne_NL': 'ANL',
        'Acne_L': 'AL'
    }

    new_labels = [
        f"{label_map[g]}\n(n={group_sizes[g]})"
        for g in present_groups
    ]

    plt.xticks(
        ticks=range(len(new_labels)),
        labels=new_labels,
        rotation=0,
        ha='center',
        fontsize=16
    )
    plt.yticks(fontsize=16)

    plt.legend(
        loc='center left',
        bbox_to_anchor=(1.0, 0.5),
        fontsize=14,
        title='Genus',
        title_fontsize=16
    )

    plt.tight_layout()
    plt.savefig(output_png_file, dpi=600)
    plt.close()


In [331]:
# Load metadata once (indexed by SampleID)
metadata_idx = metadata if metadata.index.name == 'SampleID' else metadata.set_index('SampleID')

# Get sample sets for each BIOM
biom_sample_sets = {}

for key, biom_path in biom_paths.items():
    df_tmp = load_biom_table(biom_path, metadata_path)
    biom_sample_sets[key] = set(df_tmp.columns)

# Intersection across all BIOMs
shared_samples_all = set.intersection(*biom_sample_sets.values())

# Also enforce presence in metadata
shared_samples_all = shared_samples_all.intersection(metadata_idx.index)

shared_samples_all = sorted(shared_samples_all)

print("Shared samples across all BIOMs:", len(shared_samples_all))


Shared samples across all BIOMs: 184


In [332]:
global_taxa_color_map = {}

for key, biom_path in biom_paths.items():

    df = load_biom_table(biom_path, metadata_path)

    # Enforce identical samples across BIOMs
    df = df.loc[:, shared_samples_all]
    metadata_subset = metadata_idx.loc[shared_samples_all]

    df_top_15 = collapse_top_n(df)

    output_dir = '../Figures/Main/Figure_3/'
    os.makedirs(output_dir, exist_ok=True)

    print(f"{key}: using {df.shape[1]} aligned samples")

    global_taxa_color_map = get_taxa_colors(
        df_top_15.index,
        global_taxa_color_map
    )

    plot_relative_abundance(
        df_top_15,
        metadata_subset,
        'group',
        output_dir,
        key,
        global_taxa_color_map
    )


V1-V3: using 184 aligned samples


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_24658/1785419436.py:12: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_grouped = df.groupby(metadata[group_column], axis=1).mean()


V4: using 184 aligned samples


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_24658/1785419436.py:12: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_grouped = df.groupby(metadata[group_column], axis=1).mean()
